# Multi-color takeover animation

Watch **one** run of the multi-color Moran process flood a graph, saved as a
PowerPoint-ready MP4 (H.264/yuv420p; insert with *Insert -> Video -> This Device*).

Two panels side by side: the **network**, with nodes recolored each step by the lineage
they carry, and the **lineage-size plot**, revealed left to right as the step advances.
Both share one palette, so node `k` is the same hue in both.

This is a presentation asset for a single run. The measurement, over many runs and many
graphs, lives in `multicolor_simulation.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
import networkx as nx
import imageio_ffmpeg
from IPython.display import Video

from moran_process import CppMultiColorMoranProcess as MultiColorMoranProcess
from moran_process import PopulationGraph

# Point matplotlib at the ffmpeg binary bundled with imageio-ffmpeg, so we can write
# real MP4 video without a system ffmpeg install.
plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

# Bold, high-contrast palette for slides (tab20's pastel tints wash out on a projector).
# Ordered contrast-first: the most distinct, saturated hues come before any light ones.
PRESENTATION_COLORS = [
    "#4363d8",
    "#e6194b",
    "#3cb44b",
    "#f58231",
    "#911eb4",
    "#469990",
    "#f032e6",
    "#9a6324",
    "#42d4f4",
    "#808000",
    "#800000",
    "#000075",
    "#a9a9a9",
    "#bfef45",
    "#ffe119",
    "#fabed4",
    "#dcbeff",
    "#aaffc3",
    "#ffd8b1",
    "#fffac8",
]


def lineage_palette(n):
    """Return an (n, 4) RGBA array of bold per-lineage colors (cycles if n > 20)."""
    base = np.array([mcolors.to_rgba(c) for c in PRESENTATION_COLORS])
    return base[np.arange(n) % len(base)]

In [ ]:
def animate_takeover(
    pop_graph,
    result=None,
    fps=10,
    hold_frames=12,
    max_frames=120,
    node_size=600,
    out_dir="figures",
    seed=42,
):
    """Save an MP4 of one multi-color takeover: network (left) + lineage-size plot (right).

    Pass the `result` from an earlier `sim.run(track_history=True)` to animate
    that exact run (so this matches a static plot of the same `result`); leave it
    None to run a fresh simulation on `pop_graph` internally.

    Left panel: the graph drawn once, nodes recolored each step by the lineage
    they carry. Right panel: the cell-5 lineage-size lines revealed left-to-right
    as the step advances. Long runs are subsampled to `max_frames` evenly spaced
    steps (the final takeover frame is always kept) and held for `hold_frames`
    extra frames so the result is readable before the loop restarts. Writes a
    PowerPoint-friendly H.264/yuv420p MP4 and returns its path.
    """
    if result is None:
        sim = MultiColorMoranProcess(pop_graph)
        sim.initialize_unique_colors()
        result = sim.run(track_history=True)
    history, winner = result["history"], result["winner"]
    T, n = history.shape

    # Per-step lineage sizes (same computation as cell 5), precomputed once.
    lineage_counts = np.zeros((T, n), dtype=int)
    for t in range(T):
        u, c = np.unique(history[t], return_counts=True)
        lineage_counts[t, u] = c

    # Fixed layout: reuse the factory's stored coordinates, else a spring layout.
    pos = nx.get_node_attributes(pop_graph.graph, "pos") or nx.spring_layout(
        pop_graph.graph, seed=seed
    )

    # One stable, slide-friendly color per founding lineage (shared with cell 5).
    palette = lineage_palette(n)

    # Evenly subsample long runs, but always include the final (fixed) state.
    if T > max_frames:
        idx = np.unique(np.r_[np.linspace(0, T - 1, max_frames).astype(int), T - 1])
    else:
        idx = np.arange(T)

    fig, (ax_net, ax_line) = plt.subplots(1, 2, figsize=(13, 6))

    # --- left: network, drawn once then recolored per frame ---
    ax_net.axis("off")
    nx.draw_networkx_edges(pop_graph.graph, pos, ax=ax_net, edge_color="#cccccc")
    nodes = nx.draw_networkx_nodes(
        pop_graph.graph,
        pos,
        ax=ax_net,
        node_color=palette[history[idx[0]]],
        node_size=node_size,
        edgecolors="white",
        linewidths=1.0,
    )
    net_title = ax_net.set_title("")

    # --- right: lineage-size lines, grown left-to-right ---
    lines = []
    for c in range(n):
        lw = 2.5 if c == winner else 1.0
        (ln,) = ax_line.plot(
            [],
            [],
            color=palette[c],
            linewidth=lw,
            label=f"node {c}" + (" (winner)" if c == winner else ""),
        )
        lines.append(ln)
    ax_line.set_xlim(0, max(T - 1, 1))
    ax_line.set_ylim(0, n)
    ax_line.set_xlabel("Step")
    ax_line.set_ylabel("Lineage size (# nodes)")
    ax_line.set_title(f"Lineage competition (winner = node {winner})")
    ax_line.legend(loc="upper right", ncol=2, fontsize=7)

    def update(k):
        t = idx[min(k, len(idx) - 1)]  # last frames repeat -> end-hold
        state = history[t]
        nodes.set_facecolor(palette[state])
        net_title.set_text(
            f"step {t}/{T - 1}    lineage {winner}: {np.mean(state == winner):.0%} of graph"
        )
        xs = np.arange(t + 1)
        for c, ln in enumerate(lines):
            ln.set_data(xs, lineage_counts[: t + 1, c])
        return (nodes, net_title, *lines)

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=len(idx) + hold_frames,
        interval=1000 / fps,
        blit=False,
    )

    # H.264 + yuv420p is the widely-compatible combo for PowerPoint/QuickTime; the pad
    # filter rounds odd dimensions up to even (yuv420p requires even width/height).
    writer = animation.FFMpegWriter(
        fps=fps,
        bitrate=2400,
        extra_args=["-pix_fmt", "yuv420p", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2"],
    )
    out_path = Path(out_dir) / f"{pop_graph.name}_takeover.mp4"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    anim.save(out_path, writer=writer)
    plt.close(fig)
    print(f"winner=lineage {winner}   steps={T - 1}   ->   {out_path}")
    return out_path

A grid reads especially well on a slide. Any `PopulationGraph` works; keep N small
enough that the per-lineage colors stay distinguishable.

In [ ]:
g = PopulationGraph.grid_graph(5, 5)

sim = MultiColorMoranProcess(g, seed=42)
sim.initialize_unique_colors()
result = sim.run(track_history=True)
print(f"winner = node {result['winner']}   steps = {result['steps']:,}")

mp4_path = animate_takeover(g, result)
Video(str(mp4_path), embed=True)